In [2]:
    import warnings
    warnings.filterwarnings('ignore')


In [4]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import gc

# 1. Configuration
DATASET_PATH = "Delhi_NCR_Master_DataCube_Scaled.npy"
MODEL_PATH = "Delhi_NCR_Air_Quality_Best_Model.keras"

# Actual Channel 2 (CO) scalers from your JSON
CO_MIN = 0.0195770263671875
CO_MAX = 0.11248779296875

print("Loading Data and Model....")
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
master_data = np.load(DATASET_PATH, mmap_mode='r')

train_split = int(master_data.shape[0] * 0.8)
test_indices = range(train_split, master_data.shape[0] - 8)

y_true_real_list = []
y_pred_real_list = []

print(f"Calculating scores for {len(test_indices)} test days...")

for i in test_indices:
    # Prepare Input (7-day window)
    X = master_data[i:i+7].reshape((1, 7, 141, 231, 6)).astype('float16')
    
    # Get Reality and Prediction (Scaled 0-1)
    Y_true_scaled = master_data[i+7][:, :, 2]
    Y_pred_scaled = model.predict(X, verbose=0)[0][:, :, 2]
    
    # Unscale to real-world concentration units (mol/m²)
    Y_true_real = Y_true_scaled * (CO_MAX - CO_MIN) + CO_MIN
    Y_pred_real = Y_pred_scaled * (CO_MAX - CO_MIN) + CO_MIN
    
    # We take the spatial mean of the day to evaluate temporal accuracy
    y_true_real_list.append(np.mean(Y_true_real))
    y_pred_real_list.append(np.mean(Y_pred_real))
    
    if (i - train_split) % 100 == 0:
        print(f" -> Day {i - train_split} processed...")
    
    gc.collect()

# Convert to numpy arrays for final calculation
y_true = np.array(y_true_real_list)
y_pred = np.array(y_pred_real_list)

# 2. Score Calculations
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

# MAPE calculation (Adding a tiny epsilon to avoid division by zero)
mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100

# 3. Final Output Block
print("\n" + "="*60)
print("             Conv-LSTM MODEL EVALUATION (20% TEST SET)")
print("="*60)
print(f"MAE (Mean Absolute Error)     : {mae:.6f} mol/m²")
print(f"RMSE (Root Mean Square Error) : {rmse:.6f} mol/m²")
print(f"R² Score (Coeff. of Det.)     : {r2:.4f}")
print(f"MAPE (Mean Abs. % Error)      : {mape:.2f}%")
print("="*60)

Loading Data and Model....
Calculating scores for 577 test days...
 -> Day 0 processed...
 -> Day 100 processed...
 -> Day 200 processed...
 -> Day 300 processed...
 -> Day 400 processed...
 -> Day 500 processed...

             Conv-LSTM MODEL EVALUATION (20% TEST SET)
MAE (Mean Absolute Error)     : 0.001925 mol/m²
RMSE (Root Mean Square Error) : 0.002835 mol/m²
R² Score (Coeff. of Det.)     : 0.6506
MAPE (Mean Abs. % Error)      : 4.47%


In [1]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import gc

# =====================================================================
# CUSTOM CLASS REQUIRED TO LOAD CONV-GRU WEIGHTS
# =====================================================================
class SpatiotemporalConvGRU(tf.keras.layers.Layer):
    def __init__(self, filters=32, **kwargs):
        super(SpatiotemporalConvGRU, self).__init__(**kwargs)
        self.filters = filters
        self.conv_z = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='sigmoid', name='Update_Gate')
        self.conv_r = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='sigmoid', name='Reset_Gate')
        self.conv_h = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='tanh', name='Candidate_H')

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        h = tf.zeros((batch_size, 141, 231, self.filters))
        for t in range(7):
            x_t = inputs[:, t, :, :, :] 
            concat_xh = tf.concat([x_t, h], axis=-1)
            z = self.conv_z(concat_xh)
            r = self.conv_r(concat_xh)
            concat_x_rh = tf.concat([x_t, r * h], axis=-1)
            h_tilde = self.conv_h(concat_x_rh)
            h = (1.0 - z) * h + z * h_tilde
        return h

    def get_config(self):
        config = super(SpatiotemporalConvGRU, self).get_config()
        config.update({"filters": self.filters})
        return config
# =====================================================================

# 1. Configuration
DATASET_PATH = "Delhi_NCR_Master_DataCube_Scaled.npy"
MODEL_PATH = "Delhi_NCR_ConvGRU_Baseline.keras" # <-- Update if your file name is different

# Actual Channel 2 (CO) scalers from your JSON
CO_MIN = 0.0195770263671875
CO_MAX = 0.11248779296875

print("Loading Data and ConvGRU Model....")
model = tf.keras.models.load_model(
    MODEL_PATH, 
    compile=False,
    custom_objects={"SpatiotemporalConvGRU": SpatiotemporalConvGRU}
)
master_data = np.load(DATASET_PATH, mmap_mode='r')

train_split = int(master_data.shape[0] * 0.8)
test_indices = range(train_split, master_data.shape[0] - 8)

y_true_real_list = []
y_pred_real_list = []

print(f"Calculating ConvGRU scores for {len(test_indices)} test days...")

for i in test_indices:
    X = master_data[i:i+7].reshape((1, 7, 141, 231, 6)).astype('float16')
    
    Y_true_scaled = master_data[i+7][:, :, 2]
    Y_pred_scaled = model.predict(X, verbose=0)[0][:, :, 2]
    
    Y_true_real = Y_true_scaled * (CO_MAX - CO_MIN) + CO_MIN
    Y_pred_real = Y_pred_scaled * (CO_MAX - CO_MIN) + CO_MIN
    
    y_true_real_list.append(np.mean(Y_true_real))
    y_pred_real_list.append(np.mean(Y_pred_real))
    
    if (i - train_split) % 100 == 0:
        print(f" -> Day {i - train_split} processed...")
    gc.collect()

y_true = np.array(y_true_real_list)
y_pred = np.array(y_pred_real_list)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100

print("\n" + "="*60)
print("             CONV-GRU EVALUATION (20% TEST SET)")
print("="*60)
print(f"MAE (Mean Absolute Error)     : {mae:.6f} mol/m²")
print(f"RMSE (Root Mean Square Error) : {rmse:.6f} mol/m²")
print(f"R² Score (Coeff. of Det.)     : {r2:.4f}")
print(f"MAPE (Mean Abs. % Error)      : {mape:.2f}%")
print("="*60)

I0000 00:00:1777316304.858997   68933 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777316305.015292   68933 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/yashaswi-garg/anaconda3/envs/aerosense/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
I0000 00:00:1777316309.589379   68933 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will n

Loading Data and ConvGRU Model....


/home/yashaswi-garg/anaconda3/envs/aerosense/lib/python3.10/site-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'spatiotemporal_conv_gru', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
E0000 00:00:1777316311.050501   68933 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Calculating ConvGRU scores for 577 test days...
 -> Day 0 processed...
 -> Day 100 processed...
 -> Day 200 processed...
 -> Day 300 processed...
 -> Day 400 processed...
 -> Day 500 processed...

             CONV-GRU EVALUATION (20% TEST SET)
MAE (Mean Absolute Error)     : 0.001381 mol/m²
RMSE (Root Mean Square Error) : 0.001974 mol/m²
R² Score (Coeff. of Det.)     : 0.8305
MAPE (Mean Abs. % Error)      : 3.37%


In [9]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import gc

# 1. Configuration
DATASET_PATH = "Delhi_NCR_Advanced_10Ch_Cube.npy" # <-- The 10-channel data!
MODEL_PATH = "Delhi_NCR_Advanced_ConvLSTM_Best.keras" 

# Actual Channel 2 (CO) scalers from your JSON
CO_MIN = 0.0195770263671875
CO_MAX = 0.11248779296875

print("Loading 10-Channel Data and Final ConvLSTM Model...")
# Using compile=False bypasses the need to load the custom weighted loss function for pure evaluation!
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
master_data = np.load(DATASET_PATH, mmap_mode='r')

train_split = int(master_data.shape[0] * 0.8)
test_indices = range(train_split, master_data.shape[0] - 8)

y_true_real_list = []
y_pred_real_list = []

print(f"Calculating Final Master scores for {len(test_indices)} test days...")

for i in test_indices:
    # 5D Input: 7 Days, 10 Channels (Original 6 + 4 Lagged Wind)
    X = master_data[i:i+7].reshape((1, 7, 141, 231, 10)).astype('float16')
    
    # Ground Truth: CO is still at index 2 in the DataCube
    Y_true_scaled = master_data[i+7][:, :, 2]
    
    # Prediction: The advanced model only outputs 1 channel (CO), so we use index 0!
    Y_pred_scaled = model.predict(X, verbose=0)[0][:, :, 0]
    
    # Unscale to real-world concentration units (mol/m²)
    Y_true_real = Y_true_scaled * (CO_MAX - CO_MIN) + CO_MIN
    Y_pred_real = Y_pred_scaled * (CO_MAX - CO_MIN) + CO_MIN
    
    y_true_real_list.append(np.mean(Y_true_real))
    y_pred_real_list.append(np.mean(Y_pred_real))
    
    if (i - train_split) % 100 == 0:
        print(f" -> Day {i - train_split} processed...")
    gc.collect()

y_true = np.array(y_true_real_list)
y_pred = np.array(y_pred_real_list)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100

print("\n" + "="*60)
print("       ADVANCED 10-CH CONVLSTM EVALUATION (20% TEST SET)")
print("="*60)
print(f"MAE (Mean Absolute Error)     : {mae:.6f} mol/m²")
print(f"RMSE (Root Mean Square Error) : {rmse:.6f} mol/m²")
print(f"R² Score (Coeff. of Det.)     : {r2:.4f}")
print(f"MAPE (Mean Abs. % Error)      : {mape:.2f}%")
print("="*60)

Loading 10-Channel Data and Final ConvLSTM Model...
Calculating Final Master scores for 577 test days...
 -> Day 0 processed...
 -> Day 100 processed...
 -> Day 200 processed...
 -> Day 300 processed...
 -> Day 400 processed...
 -> Day 500 processed...

       ADVANCED 10-CH CONVLSTM EVALUATION (20% TEST SET)
MAE (Mean Absolute Error)     : 0.001638 mol/m²
RMSE (Root Mean Square Error) : 0.002203 mol/m²
R² Score (Coeff. of Det.)     : 0.7890
MAPE (Mean Abs. % Error)      : 4.06%


In [1]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import gc

# 1. Configuration
DATASET_PATH = "Delhi_NCR_Master_DataCube_Scaled.npy"
MODEL_PATH = "Delhi_NCR_UNet_Best.keras" 

# Actual Channel 2 (CO) scalers from your JSON
CO_MIN = 0.0195770263671875
CO_MAX = 0.11248779296875

print("Loading Data and U-Net Model....")
# No custom objects needed for a standard U-Net
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
master_data = np.load(DATASET_PATH, mmap_mode='r')

train_split = int(master_data.shape[0] * 0.8)
test_indices = range(train_split, master_data.shape[0] - 8)

y_true_real_list = []
y_pred_real_list = []

print(f"Calculating U-Net scores for {len(test_indices)} test days...")

for i in test_indices:
    # Get the 7-day lookback data: shape (7, 141, 231, 6)
    raw_X = master_data[i:i+7]
    
    # CRITICAL U-NET RESHAPE: Flattening time (7) and channels (6) into 42 channels
    # We transpose to (141, 231, 7, 6) then reshape to (1, 141, 231, 42)
    # This ensures the spatial grid remains perfectly intact while combining the features.
    X = np.transpose(raw_X, (1, 2, 0, 3)).reshape((1, 141, 231, 42)).astype('float16')
    
    # Ground Truth: CO is still at index 2
    Y_true_scaled = master_data[i+7][:, :, 2]
    
    # U-Net Prediction: Output should match the 6-channel target, we want index 2 (CO)
    Y_pred_scaled = model.predict(X, verbose=0)[0][:, :, 2]
    
    # Unscale to real-world concentration units (mol/m²)
    Y_true_real = Y_true_scaled * (CO_MAX - CO_MIN) + CO_MIN
    Y_pred_real = Y_pred_scaled * (CO_MAX - CO_MIN) + CO_MIN
    
    y_true_real_list.append(np.mean(Y_true_real))
    y_pred_real_list.append(np.mean(Y_pred_real))
    
    if (i - train_split) % 100 == 0:
        print(f" -> Day {i - train_split} processed...")
    gc.collect()

y_true = np.array(y_true_real_list)
y_pred = np.array(y_pred_real_list)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100

print("\n" + "="*60)
print("             U-NET BASELINE EVALUATION (20% TEST SET)")
print("="*60)
print(f"MAE (Mean Absolute Error)     : {mae:.6f} mol/m²")
print(f"RMSE (Root Mean Square Error) : {rmse:.6f} mol/m²")
print(f"R² Score (Coeff. of Det.)     : {r2:.4f}")
print(f"MAPE (Mean Abs. % Error)      : {mape:.2f}%")
print("="*60)

I0000 00:00:1776706120.029493    9310 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776706120.540348    9310 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/yashaswi-garg/anaconda3/envs/aerosense/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
I0000 00:00:1776706123.338771    9310 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will n

Loading Data and U-Net Model....


E0000 00:00:1776706125.216224    9310 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Calculating U-Net scores for 577 test days...
 -> Day 0 processed...
 -> Day 100 processed...
 -> Day 200 processed...
 -> Day 300 processed...
 -> Day 400 processed...
 -> Day 500 processed...

             U-NET BASELINE EVALUATION (20% TEST SET)
MAE (Mean Absolute Error)     : 0.001996 mol/m²
RMSE (Root Mean Square Error) : 0.002754 mol/m²
R² Score (Coeff. of Det.)     : 0.6704
MAPE (Mean Abs. % Error)      : 4.74%


In [2]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import gc

# 1. Configuration
DATASET_PATH = "Delhi_NCR_Master_DataCube_Scaled.npy"
MODEL_PATH = "Delhi_NCR_ResNet50_Best.keras" 

# Actual Channel 2 (CO) scalers
CO_MIN = 0.0195770263671875
CO_MAX = 0.11248779296875

print("Loading Data and ResNet-50 Model....")
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
master_data = np.load(DATASET_PATH, mmap_mode='r')

train_split = int(master_data.shape[0] * 0.8)
test_indices = range(train_split, master_data.shape[0] - 8)

y_true_real_list = []
y_pred_real_list = []

print(f"Calculating ResNet-50 scores for {len(test_indices)} test days...")

for i in test_indices:
    # Get the 7-day lookback data: shape (7, 141, 231, 6)
    raw_X = master_data[i:i+7]
    
    # CRITICAL RESHAPE: Flattening time (7) and channels (6) into 42 channels
    # Transpose to (141, 231, 7, 6) then reshape to (1, 141, 231, 42)
    X = np.transpose(raw_X, (1, 2, 0, 3)).reshape((1, 141, 231, 42)).astype('float16')
    
    # Ground Truth: CO is at index 2
    Y_true_scaled = master_data[i+7][:, :, 2]
    
    # Prediction: Output matches 6-channel target, grab index 2 (CO)
    Y_pred_scaled = model.predict(X, verbose=0)[0][:, :, 2]
    
    # Unscale to real-world concentration units (mol/m²)
    Y_true_real = Y_true_scaled * (CO_MAX - CO_MIN) + CO_MIN
    Y_pred_real = Y_pred_scaled * (CO_MAX - CO_MIN) + CO_MIN
    
    y_true_real_list.append(np.mean(Y_true_real))
    y_pred_real_list.append(np.mean(Y_pred_real))
    
    if (i - train_split) % 100 == 0:
        print(f" -> Day {i - train_split} processed...")
    
    # Clean up RAM
    del raw_X, X, Y_true_scaled, Y_pred_scaled, Y_true_real, Y_pred_real
    gc.collect()

y_true = np.array(y_true_real_list)
y_pred = np.array(y_pred_real_list)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100

print("\n" + "="*60)
print("          RESNET-50 BASELINE EVALUATION (20% TEST SET)")
print("="*60)
print(f"MAE (Mean Absolute Error)     : {mae:.6f} mol/m²")
print(f"RMSE (Root Mean Square Error) : {rmse:.6f} mol/m²")
print(f"R² Score (Coeff. of Det.)     : {r2:.4f}")
print(f"MAPE (Mean Abs. % Error)      : {mape:.2f}%")
print("="*60)

Loading Data and ResNet-50 Model....


E0000 00:00:1776954150.695420    6430 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Calculating ResNet-50 scores for 577 test days...
 -> Day 0 processed...
 -> Day 100 processed...
 -> Day 200 processed...
 -> Day 300 processed...
 -> Day 400 processed...
 -> Day 500 processed...

          RESNET-50 BASELINE EVALUATION (20% TEST SET)
MAE (Mean Absolute Error)     : 0.001952 mol/m²
RMSE (Root Mean Square Error) : 0.002679 mol/m²
R² Score (Coeff. of Det.)     : 0.6879
MAPE (Mean Abs. % Error)      : 4.62%


In [3]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import gc

# 1. Configuration
DATASET_PATH = "Delhi_NCR_Master_DataCube_Scaled.npy"
MODEL_PATH = "Delhi_NCR_ResNet101_Best.keras" 

# Actual Channel 2 (CO) scalers
CO_MIN = 0.0195770263671875
CO_MAX = 0.11248779296875

print("Loading Data and ResNet-101 Model....")
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
master_data = np.load(DATASET_PATH, mmap_mode='r')

train_split = int(master_data.shape[0] * 0.8)
test_indices = range(train_split, master_data.shape[0] - 8)

y_true_real_list = []
y_pred_real_list = []

print(f"Calculating ResNet-101 scores for {len(test_indices)} test days...")

for i in test_indices:
    raw_X = master_data[i:i+7]
    
    # CRITICAL RESHAPE: Flattening time (7) and channels (6) into 42 channels
    X = np.transpose(raw_X, (1, 2, 0, 3)).reshape((1, 141, 231, 42)).astype('float16')
    
    Y_true_scaled = master_data[i+7][:, :, 2]
    Y_pred_scaled = model.predict(X, verbose=0)[0][:, :, 2]
    
    Y_true_real = Y_true_scaled * (CO_MAX - CO_MIN) + CO_MIN
    Y_pred_real = Y_pred_scaled * (CO_MAX - CO_MIN) + CO_MIN
    
    y_true_real_list.append(np.mean(Y_true_real))
    y_pred_real_list.append(np.mean(Y_pred_real))
    
    if (i - train_split) % 100 == 0:
        print(f" -> Day {i - train_split} processed...")
    
    del raw_X, X, Y_true_scaled, Y_pred_scaled, Y_true_real, Y_pred_real
    gc.collect()

y_true = np.array(y_true_real_list)
y_pred = np.array(y_pred_real_list)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100

print("\n" + "="*60)
print("         RESNET-101 BASELINE EVALUATION (20% TEST SET)")
print("="*60)
print(f"MAE (Mean Absolute Error)     : {mae:.6f} mol/m²")
print(f"RMSE (Root Mean Square Error) : {rmse:.6f} mol/m²")
print(f"R² Score (Coeff. of Det.)     : {r2:.4f}")
print(f"MAPE (Mean Abs. % Error)      : {mape:.2f}%")
print("="*60)

Loading Data and ResNet-101 Model....
Calculating ResNet-101 scores for 577 test days...
 -> Day 0 processed...
 -> Day 100 processed...
 -> Day 200 processed...
 -> Day 300 processed...
 -> Day 400 processed...
 -> Day 500 processed...

         RESNET-101 BASELINE EVALUATION (20% TEST SET)
MAE (Mean Absolute Error)     : 0.002102 mol/m²
RMSE (Root Mean Square Error) : 0.002909 mol/m²
R² Score (Coeff. of Det.)     : 0.6323
MAPE (Mean Abs. % Error)      : 4.94%


In [4]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import gc

# ==========================================
# 1. Configuration
# ==========================================
DATASET_PATH = "Delhi_NCR_Master_DataCube_Scaled.npy"
MODEL_PATH = "Delhi_NCR_AlexNet_Best.keras" # <-- Pointed at AlexNet!

# Actual Channel 2 (CO) scalers from your dataset
CO_MIN = 0.0195770263671875
CO_MAX = 0.11248779296875

print("Loading Data and AlexNet Model....")
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
master_data = np.load(DATASET_PATH, mmap_mode='r')

train_split = int(master_data.shape[0] * 0.8)
test_indices = range(train_split, master_data.shape[0] - 8)

y_true_real_list = []
y_pred_real_list = []

print(f"Calculating AlexNet scores for {len(test_indices)} test days...")

# ==========================================
# 2. Evaluation Loop
# ==========================================
for i in test_indices:
    # Get the 7-day lookback data: shape (7, 141, 231, 6)
    raw_X = master_data[i:i+7]
    
    # CRITICAL RESHAPE: Flattening time (7) and channels (6) into 42 channels
    # Transpose to (141, 231, 7, 6) then reshape to (1, 141, 231, 42)
    X = np.transpose(raw_X, (1, 2, 0, 3)).reshape((1, 141, 231, 42)).astype('float16')
    
    # Ground Truth: CO is at index 2
    Y_true_scaled = master_data[i+7][:, :, 2]
    
    # Prediction: Output matches 6-channel target, grab index 2 (CO)
    Y_pred_scaled = model.predict(X, verbose=0)[0][:, :, 2]
    
    # Unscale to real-world concentration units (mol/m²)
    Y_true_real = Y_true_scaled * (CO_MAX - CO_MIN) + CO_MIN
    Y_pred_real = Y_pred_scaled * (CO_MAX - CO_MIN) + CO_MIN
    
    y_true_real_list.append(np.mean(Y_true_real))
    y_pred_real_list.append(np.mean(Y_pred_real))
    
    if (i - train_split) % 100 == 0:
        print(f" -> Day {i - train_split} processed...")
    
    # Clean up RAM to prevent Jupyter crashes
    del raw_X, X, Y_true_scaled, Y_pred_scaled, Y_true_real, Y_pred_real
    gc.collect()

# ==========================================
# 3. Final Metrics
# ==========================================
y_true = np.array(y_true_real_list)
y_pred = np.array(y_pred_real_list)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100

print("\n" + "="*60)
print("            ALEXNET BASELINE EVALUATION (20% TEST SET)")
print("="*60)
print(f"MAE (Mean Absolute Error)     : {mae:.6f} mol/m²")
print(f"RMSE (Root Mean Square Error) : {rmse:.6f} mol/m²")
print(f"R² Score (Coeff. of Det.)     : {r2:.4f}")
print(f"MAPE (Mean Abs. % Error)      : {mape:.2f}%")
print("="*60)

Loading Data and AlexNet Model....
Calculating AlexNet scores for 577 test days...
 -> Day 0 processed...
 -> Day 100 processed...
 -> Day 200 processed...
 -> Day 300 processed...
 -> Day 400 processed...
 -> Day 500 processed...

            ALEXNET BASELINE EVALUATION (20% TEST SET)
MAE (Mean Absolute Error)     : 0.001941 mol/m²
RMSE (Root Mean Square Error) : 0.002538 mol/m²
R² Score (Coeff. of Det.)     : 0.7200
MAPE (Mean Abs. % Error)      : 4.78%
